In [1]:
import duckdb
import numpy as np
import pandas as pd

In [2]:
#   Analysing a particular scenario
OBSERVATIONS_PATH = "/Users/saranyapal/Developer/tracklet-tracker/data/candidate-packet/data/train/scenario_001/observations.jsonl"
LABELS_PATH = "/Users/saranyapal/Developer/tracklet-tracker/data/candidate-packet/data/train/scenario_001/labels.jsonl"
SUPPORTING_PATH = "/Users/saranyapal/Developer/tracklet-tracker/data/candidate-packet/data/train/scenario_001/supporting.jsonl"

#   JSON
SNAPSHOT_PATH = "/Users/saranyapal/Developer/tracklet-tracker/data/candidate-packet/data/train/scenario_001/snapshot.json"

In [3]:
def process_sql_query(query: str):
    return duckdb.query(query).df()


def deduplicating_df(df: pd.DataFrame, subset: str) -> pd.DataFrame:
    df = df.drop_duplicates(subset=[subset], keep=False)
    
    return df.reset_index(drop=True)


def process_observations(observations_path: str) -> pd.DataFrame:
    observations_df = pd.read_json(observations_path, lines=True)
    observations_df = deduplicating_df(observations_df, "observation_id")

    #   Removing all rows with invalid latitude and longitude
    observations_df = observations_df[
        observations_df["latitude"].between(-90.0, 90.0) &
        observations_df["longitude"].between(-180.0, 180.0)
    ]

    #   Converting to DateTime for splink
    observations_df['observed_at'] = pd.to_datetime(observations_df['observed_at'])
    observations_df['received_at'] = pd.to_datetime(observations_df['received_at'])

    attributes_df = pd.json_normalize(observations_df["attributes"]).add_prefix("observ_")
    observations_df = pd.concat(
        [observations_df.drop(columns=["attributes"]), attributes_df],
        axis=1
    )


    #   Adjusted for the difference in speed units.....(as warned in the document)
    if 'observ_spd' in observations_df.columns:
        # Create a mask for rows where the source is 'feed-a' and speed is not null
        is_feed_a = (observations_df['source'] == 'feed-a') & (observations_df['observ_spd'].notna())
        
        # Convert those specific rows from knots to m/s
        observations_df.loc[is_feed_a, 'observ_spd'] = observations_df.loc[is_feed_a, 'observ_spd'] * 0.514444

    return observations_df




def process_supporting(supporting_path: str) -> pd.DataFrame:
    supporting_df = pd.read_json(supporting_path, lines=True)
    supporting_df = deduplicating_df(supporting_df, "record_id")

    if 'received_at' in supporting_df.columns:
        supporting_df['received_at'] = pd.to_datetime(supporting_df['received_at'])


    if "attributes" in supporting_df.columns:
        attributes_df_2 = pd.json_normalize(supporting_df["attributes"]).add_prefix("supporting_")
        supporting_df = pd.concat(
            [supporting_df.drop(columns=["attributes"]), attributes_df_2],
            axis=1
        )

    return supporting_df




def read_scenario(observations_path: str, labels_path: str, supporting_path: str, snapshot_path: str):
    observations_df = process_observations(observations_path)    
    labels_df = pd.read_json(labels_path, lines=True)
    supporting_df = process_supporting(supporting_path)
    snapshot_df = pd.read_json(snapshot_path)       

    return observations_df, labels_df, supporting_df, snapshot_df





def table_join(observations_df, supporting_df):
    merged_df = pd.merge(
        observations_df,
        supporting_df,
        on="observation_id",
        how="left",
        suffixes=('', '_supp')
    )

    #   Sorting by tracklet_id and time (Probably the most important step)
    merged_df = merged_df.sort_values(by=["tracklet_id", "observed_at"]).reset_index(drop=True)
    return merged_df



def aggregation(merged_df: pd.DataFrame) -> pd.DataFrame:
    #   The clarifying document mentions:
    #   "Each tracklet contains observations from exactly one aircraft. 
    #   Its membership is fixed. You do not need to split tracklets, merge duplicate tracklets, 
    #   or question whether observations within a tracklet belong together."
    agg_funcs = {
        'observed_at': ['first', 'last'],
        'latitude': ['first', 'last'],
        'longitude': ['first', 'last'],
        
        # Rescue Categorical/ID features
        'source': ['first'],
        'observ_ref': ['first'],
        
        # Rescue Numerical physics features (ignoring NaNs automatically)
        'observ_spd': ['mean'],
        'observ_hdg': ['mean'],
        'supporting_spd': ['mean']
    }

    tracklets_summary = merged_df.groupby('tracklet_id').agg(agg_funcs).reset_index()
    tracklets_summary.columns = [
        'tracklet_id', 
        'start_time', 'end_time', 
        'start_lat', 'end_lat', 
        'start_lon', 'end_lon',
        'source',         # rescued
        'observ_ref',     # rescued
        'observ_spd',     # rescued
        'observ_hdg',     # rescued
        'supporting_spd'  # rescued
    ]

    ## Additional work:
    #       For now, we have not even used the supporting_df's informations. Need to use them for sure
    #       Can use the extra attributes 

    return tracklets_summary

In [4]:
observations_df, labels_df, supporting_df, snapshot_df = read_scenario(
    OBSERVATIONS_PATH,
    LABELS_PATH,
    SUPPORTING_PATH,
    SNAPSHOT_PATH
)

print(observations_df.shape)
print(labels_df.shape)
print(supporting_df.shape)
print(snapshot_df.shape)

(302, 18)
(24, 2)
(180, 8)
(3, 10)


In [5]:
master_df = table_join(observations_df, supporting_df)
aggregated_df = aggregation(master_df)

print(master_df.shape)
print(aggregated_df.shape)

(302, 25)
(24, 12)


In [6]:
master_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 302 entries, 0 to 301
Data columns (total 25 columns):
 #   Column            Non-Null Count  Dtype              
---  ------            --------------  -----              
 0   observation_id    302 non-null    str                
 1   tracklet_id       302 non-null    str                
 2   observed_at       302 non-null    datetime64[us, UTC]
 3   received_at       302 non-null    datetime64[us, UTC]
 4   latitude          302 non-null    float64            
 5   longitude         302 non-null    float64            
 6   source            302 non-null    str                
 7   altitude_ft       239 non-null    float64            
 8   observ_c2         76 non-null     float64            
 9   observ_ref        201 non-null    str                
 10  observ_q          277 non-null    object             
 11  observ_node       302 non-null    str                
 12  observ_r7         302 non-null    float64            
 13  observ_v        

In [7]:
#   Important Notes:

#   From observations_df:
#       1. unique_observation_count: 312
#       2. observation_row_count: 322
#       3. unique_tracklet_count: 24


#   From supporting_df:
#       1. unique_observation_count: 186

#   From labels_df:
#       1. unique_tracklet_count: 24

## All Observations:

1. [DONE] This indicates that some observations don't have supporting data (meaning a standard left join is the expected behavior.)   

2.  This also indicates that all observations do have their corresponding labels (which is just the successor_tracklet_id: next_tracklet_id) -> This represents the next link   

3. (Common sense) Drop IDs before ML: While you need tracklet_id, observation_id, and scenario_id to join and group the data, you must not use them as predictive features for your ML model.   

4. [DONE] Duplicates must be removed from observations_df and supporting_df!! Before you do the left join, you must deduplicate the supporting.jsonl records using their record_id. (Just like you deduplicate observations using observation_id).   

5. [DONE] If the observation_id is only duplicate, other fields are not -> we must remove both these     
 
6. [DONE] Coordinate Validation: You must validate coordinate ranges. If a latitude is out of bounds (e.g., greater than 90 or less than -90), you must reject that row. There is actually a specific test file mentioned (fixtures/invalid_coordinate.jsonl) designed to check if your code catches this.    

7. Missing Altitude: The document explicitly warns: "Do not silently treat missing values as zero." Altitude can be null. If you fill it with a 0, your model might think the aircraft suddenly dove to sea level. You need to handle it explicitly (e.g., using interpolation, maintaining it as a NaN, or dropping it if it's too sparse).
8. Strict Separation of Time: Ensure you are strictly using observed_at to calculate physical things like distance and speed. received_at is only when the server received the data—you will only need received_at later when you do the "staged demonstration" to simulate delayed data.


In [8]:
aggregated_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 24 entries, 0 to 23
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype              
---  ------          --------------  -----              
 0   tracklet_id     24 non-null     str                
 1   start_time      24 non-null     datetime64[us, UTC]
 2   end_time        24 non-null     datetime64[us, UTC]
 3   start_lat       24 non-null     float64            
 4   end_lat         24 non-null     float64            
 5   start_lon       24 non-null     float64            
 6   end_lon         24 non-null     float64            
 7   source          24 non-null     str                
 8   observ_ref      16 non-null     str                
 9   observ_spd      16 non-null     float64            
 10  observ_hdg      8 non-null      float64            
 11  supporting_spd  24 non-null     float64            
dtypes: datetime64[us, UTC](2), float64(7), str(3)
memory usage: 3.7 KB


In [9]:
aggregated_df.head()

,tracklet_id,start_time,end_time,start_lat,end_lat,start_lon,end_lon,source,observ_ref,observ_spd,observ_hdg,supporting_spd
0,tracklet-03b4b60d-f1d1-be22-62ea-070dd9b06819,2026-01-10 00:11:38.186+00:00,2026-01-10 00:13:46.804+00:00,35.313578,35.238779,-108.546265,-108.774255,feed-a,F51D6A,173.406968,NaN,49.000
1,tracklet-054df159-4c90-c17e-91a4-1ed8c483c568,2026-01-10 00:00:26.670+00:00,2026-01-10 00:02:24.451+00:00,35.671734,35.601289,-107.336176,-107.553976,feed-a,DC8A69,181.304707,NaN,53.000
2,tracklet-0d60a69d-4493-259d-deb2-9d191a81dde6,2026-01-10 00:07:49.415+00:00,2026-01-10 00:09:21.027+00:00,35.378941,35.315701,-107.897465,-108.016967,feed-a,20B3E3,164.860653,NaN,37.750
3,tracklet-25a03cd1-b106-0bba-6635-f770f85253dd,2026-01-10 00:07:44.230+00:00,2026-01-10 00:09:41.900+00:00,35.300919,35.206605,-107.972739,-108.147571,feed-b,6A560D,161.613846,NaN,40.500
4,tracklet-27f73aa2-d32d-3652-9edb-b41e281835c3,2026-01-10 00:11:42.560+00:00,2026-01-10 00:13:10.105+00:00,35.041899,34.959947,-108.285103,-108.411172,feed-b,2817A5,212.055000,NaN,42.875


### Feature Preparation:

In [10]:
# Step 1: Smarter Aggregation (Tracklet Level)
# When you group by tracklet_id, don't just grab the first and last points. Calculate internal physics for the tracklet so you know how the aircraft was behaving before the gap.

# A. Boundary Features (The Basics):
# 1. start_time, end_time, start_lat, end_lat, start_lon, end_lon (using .first() and .last()).
# 2. Altitude Extraction (239/302 non-nulls):
# 3. start_alt and end_alt. (Forward fill or backward fill internally if the exact first/last point is null, otherwise leave as NaN).
# 4. Internal Kinematics (How is it flying?):
# 5. internal_duration: end_time - start_time
# 6. internal_distance: Haversine distance between start and end points.
# 7. internal_speed: internal_distance / internal_duration. (This is much safer to use than observ_spd because you know it's in meters/second. The SOURCE_NOTES.md warns that observ_spd has mixed units across feeds).
# 8. internal_bearing: Calculate the compass bearing (0-360 degrees) from the start coordinate to the end coordinate. This tells you which direction the plane is pointing when it disappears.


# B. Identity Extraction (observ_ref):

# 201 out of 302 rows have observ_ref. The source notes mention this is a "reference-like value" (likely a transponder code).
# Aggregate this by grabbing the mode (most frequent value) for the tracklet.

# Drop the Noise: Drop observ_c2, observ_r7, observ_v, observ_tag, supporting_a1, supporting_b4, supporting_rx. Without documentation, these are dangerous to feed to an ML model because they might just be random sensor hardware IDs.



# Step 2: The Pairwise Features (The Magic for XGBoost)
# Now, you create a new dataframe of Candidate Pairs. For every Tracklet A, pair it with a candidate Tracklet B where B.start_time > A.end_time.

# Here is the exact list of features you should calculate for each (A, B) row to feed into XGBoost:

# Temporal & Spatial Gaps:

# gap_time: B.start_time - A.end_time (in seconds).

# gap_distance: Haversine distance from A.end_lat/lon to B.start_lat/lon.

# Kinematic Plausibility (The most important features):

# required_speed: gap_distance / gap_time. (XGBoost will easily learn that if this is > 300 m/s, it should predict 0).

# speed_change_A: abs(required_speed - A.internal_speed). Does the aircraft have to massively speed up or slow down in the blind spot to reach Tracklet B?

# required_climb_rate: (B.start_alt - A.end_alt) / gap_time. If this requires the plane to climb 10,000 feet in 10 seconds, it's not the same plane.

# Directional Alignment (The "Game Changer"):

# gap_bearing: The compass bearing from A.end_lat/lon to B.start_lat/lon.

# turn_angle_penalty: The absolute angular difference between A.internal_bearing and gap_bearing.

# Why this is genius: If Tracklet A is flying North (0°), and Tracklet B is located 50 miles East (90°) just 2 minutes later, the turn_angle_penalty is 90°. XGBoost will learn that commercial planes don't make instantaneous 90-degree turns. If it points straight at B, the penalty is 0°.

# Categorical / Identity Match:

# ref_match: 1 if A.observ_ref == B.observ_ref (and both are not null), else 0.

# cross_source: 1 if A and B came from different feeds, else 0. (Sometimes planes switch radar feeds as they move).

In [11]:
def haversine_distance(lat1, lon1, lat2, lon2):
    """Calculates the great-circle distance between two points in meters."""
    # FIX: Force Pandas Series into flat 1D NumPy arrays to prevent index/broadcasting errors
    lat1 = np.asarray(lat1, dtype=float)
    lon1 = np.asarray(lon1, dtype=float)
    lat2 = np.asarray(lat2, dtype=float)
    lon2 = np.asarray(lon2, dtype=float)
    
    R = 6371000.0  # Radius of earth in meters
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    
    a = np.sin(dphi/2.0)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda/2.0)**2
    return 2.0 * R * np.arctan2(np.sqrt(a), np.sqrt(1.0 - a))

def calculate_bearing(lat1, lon1, lat2, lon2):
    """Calculates the compass bearing (0-360 degrees) from point 1 to point 2."""
    # FIX: Force Pandas Series into flat 1D NumPy arrays 
    lat1 = np.asarray(lat1, dtype=float)
    lon1 = np.asarray(lon1, dtype=float)
    lat2 = np.asarray(lat2, dtype=float)
    lon2 = np.asarray(lon2, dtype=float)
    
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    lambda1, lambda2 = np.radians(lon1), np.radians(lon2)
    dlambda = lambda2 - lambda1
    
    y = np.sin(dlambda) * np.cos(phi2)
    x = np.cos(phi1) * np.sin(phi2) - np.sin(phi1) * np.cos(phi2) * np.cos(dlambda)
    
    bearing = np.degrees(np.arctan2(y, x))
    return (bearing + 360.0) % 360.0

In [12]:
def prepare_features_and_labels(tracklets_summary: pd.DataFrame, labels_df: pd.DataFrame) -> pd.DataFrame:
    """
    Generates all candidate pairs (A -> B), computes physical relationship features,
    and attaches binary targets based on the labels.
    """
    
    # 1. Create a Cartesian product (cross join) of the summary table with itself.
    df_a = tracklets_summary.add_prefix('A_')
    df_b = tracklets_summary.add_prefix('B_')
    
    # Perform a cross join (every tracklet against every other tracklet)
    pairs = pd.merge(df_a.assign(key=1), df_b.assign(key=1), on='key').drop('key', axis=1)
    
    # 2. Blocking/Filtering: Remove impossible pairs FIRST to save computation
    # - A tracklet cannot follow itself.
    # - Tracklet B must start strictly AFTER Tracklet A ends.
    pairs = pairs[pairs['A_tracklet_id'] != pairs['B_tracklet_id']]
    pairs = pairs[pairs['B_start_time'] > pairs['A_end_time']].copy()
    
    # 3. Calculate Base Relational Features
    # Time gap in seconds
    pairs['gap_time_seconds'] = (pairs['B_start_time'] - pairs['A_end_time']).dt.total_seconds()
    
    # Spatial distance across the gap in meters
    pairs['gap_distance_meters'] = haversine_distance(
        pairs['A_end_lat'], pairs['A_end_lon'], 
        pairs['B_start_lat'], pairs['B_start_lon']
    )
    
    # Kinematic plausibility (Required speed in meters/second)
    # Adding a tiny epsilon to gap_time to prevent division by zero
    pairs['required_speed_mps'] = pairs['gap_distance_meters'] / (pairs['gap_time_seconds'] + 1e-6)
    
    # Gap Bearing (The direction the gap points)
    pairs['gap_bearing'] = calculate_bearing(
        pairs['A_end_lat'], pairs['A_end_lon'], 
        pairs['B_start_lat'], pairs['B_start_lon']
    )

    # 4. Advanced/Attribute Features (Now that required_speed_mps exists)
    
    # Identity and Source Matching
    if 'A_observ_ref' in pairs.columns:
        pairs['ref_match_exact'] = (
            (pairs['A_observ_ref'] == pairs['B_observ_ref']) & 
            (pairs['A_observ_ref'].notna())
        ).astype(int)

    if 'A_source' in pairs.columns:
        pairs['is_same_source'] = (pairs['A_source'] == pairs['B_source']).astype(int)

    # Speed consistency checks (Use absolute difference to see if aircraft massively changed speed)
    if 'A_observ_spd' in pairs.columns:
        pairs['speed_diff_A_observ'] = abs(pairs['required_speed_mps'] - pairs['A_observ_spd'])
        pairs['speed_diff_B_observ'] = abs(pairs['required_speed_mps'] - pairs['B_observ_spd'])

    if 'A_supporting_spd' in pairs.columns:
        pairs['telemetry_speed_diff'] = abs(pairs['A_supporting_spd'] - pairs['B_supporting_spd'])

    # 5. Attach the Ground Truth Labels (The Target Variable)
    # Rename labels to match pair structure for a clean left join
    truth_df = labels_df[['tracklet_id', 'successor_tracklet_id']].copy()
    truth_df = truth_df.rename(columns={
        'tracklet_id': 'A_tracklet_id', 
        'successor_tracklet_id': 'B_tracklet_id'
    })
    
    # Add a column so we know which pairs matched the truth
    truth_df['target'] = 1
    
    # Left join the truth onto our pairs
    final_dataset = pd.merge(
        pairs, 
        truth_df, 
        on=['A_tracklet_id', 'B_tracklet_id'], 
        how='left'
    )
    
    # Any pair that wasn't in the truth table gets a 0 (False link)
    final_dataset['target'] = final_dataset['target'].fillna(0).astype(int)
    
    # 6. Clean up the dataframe for the ML model
    # Drop timestamps as ML models can't ingest raw datetimes directly
    cols_to_drop = ['A_start_time', 'A_end_time', 'B_start_time', 'B_end_time']
    final_dataset = final_dataset.drop(columns=cols_to_drop, errors='ignore')
    
    return final_dataset

In [ ]:
master_df = table_join(observations_df, supporting_df)
aggregated_df = aggregation(master_df)

ml_dataset = prepare_features_and_labels(aggregated_df, labels_df)

X = ml_dataset.drop(columns=['target', 'A_tracklet_id', 'B_tracklet_id'])
y = ml_dataset['target']

In [14]:
ml_dataset.info()

<class 'pandas.DataFrame'>
RangeIndex: 216 entries, 0 to 215
Data columns (total 30 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   A_tracklet_id         216 non-null    str    
 1   A_start_lat           216 non-null    float64
 2   A_end_lat             216 non-null    float64
 3   A_start_lon           216 non-null    float64
 4   A_end_lon             216 non-null    float64
 5   A_source              216 non-null    str    
 6   A_observ_ref          138 non-null    str    
 7   A_observ_spd          138 non-null    float64
 8   A_observ_hdg          78 non-null     float64
 9   A_supporting_spd      216 non-null    float64
 10  B_tracklet_id         216 non-null    str    
 11  B_start_lat           216 non-null    float64
 12  B_end_lat             216 non-null    float64
 13  B_start_lon           216 non-null    float64
 14  B_end_lon             216 non-null    float64
 15  B_source              216 non-null

In [15]:
ml_dataset.head()

,A_tracklet_id,A_start_lat,A_end_lat,A_start_lon,A_end_lon,A_source,A_observ_ref,A_observ_spd,A_observ_hdg,A_supporting_spd,...,gap_time_seconds,gap_distance_meters,required_speed_mps,gap_bearing,ref_match_exact,is_same_source,speed_diff_A_observ,speed_diff_B_observ,telemetry_speed_diff,target
0,tracklet-054df159-4c90-c17e-91a4-1ed8c483c568,35.671734,35.601289,-107.336176,-107.553976,feed-a,DC8A69,181.304707,NaN,53.0,...,553.735,95398.564599,172.281984,250.694544,0,1,9.022723,1.124984,4.000000,0
1,tracklet-054df159-4c90-c17e-91a4-1ed8c483c568,35.671734,35.601289,-107.336176,-107.553976,feed-a,DC8A69,181.304707,NaN,53.0,...,324.964,39728.696485,122.255685,231.614246,0,1,59.049023,42.604969,15.250000,0
2,tracklet-054df159-4c90-c17e-91a4-1ed8c483c568,35.671734,35.601289,-107.336176,-107.553976,feed-a,DC8A69,181.304707,NaN,53.0,...,319.779,50540.482303,158.048159,228.757163,0,0,23.256549,3.565687,12.500000,0
3,tracklet-054df159-4c90-c17e-91a4-1ed8c483c568,35.671734,35.601289,-107.336176,-107.553976,feed-a,DC8A69,181.304707,NaN,53.0,...,558.109,90933.325168,162.931121,227.052662,0,0,18.373586,49.123879,10.125000,0
4,tracklet-054df159-4c90-c17e-91a4-1ed8c483c568,35.671734,35.601289,-107.336176,-107.553976,feed-a,DC8A69,181.304707,NaN,53.0,...,318.097,37943.051592,119.281387,229.261724,0,1,62.023320,27.232264,0.083333,0


### Checking the inputs and outputs now:

A Quick Review of Your DatasetThe Size (216 rows): 

This confirms the Cartesian product and filtering worked perfectly. With 24 tracklets, a full cross-join would be $24 \times 24 = 576$ pairs. Dropping the impossible pairs (time-travelers and self-links) brought it down to a highly optimized 216 realistic candidate pairs.The Target (216 non-null): You have successfully attached your labels_df. Out of these 216 rows, only ~18 will have target == 1 (the true successor links), and the rest will be 0.The Features: You have beautifully engineered the physical constraints (required_speed_mps, gap_bearing, gap_distance_meters) alongside the identity metadata (ref_match_exact, is_same_source).

In [18]:
X.shape

(216, 27)

In [17]:
X.info()

<class 'pandas.DataFrame'>
RangeIndex: 216 entries, 0 to 215
Data columns (total 27 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   A_start_lat           216 non-null    float64
 1   A_end_lat             216 non-null    float64
 2   A_start_lon           216 non-null    float64
 3   A_end_lon             216 non-null    float64
 4   A_source              216 non-null    str    
 5   A_observ_ref          138 non-null    str    
 6   A_observ_spd          138 non-null    float64
 7   A_observ_hdg          78 non-null     float64
 8   A_supporting_spd      216 non-null    float64
 9   B_start_lat           216 non-null    float64
 10  B_end_lat             216 non-null    float64
 11  B_start_lon           216 non-null    float64
 12  B_end_lon             216 non-null    float64
 13  B_source              216 non-null    str    
 14  B_observ_ref          150 non-null    str    
 15  B_observ_spd          150 non-null

In [19]:
X.head()

,A_start_lat,A_end_lat,A_start_lon,A_end_lon,A_source,A_observ_ref,A_observ_spd,A_observ_hdg,A_supporting_spd,B_start_lat,...,B_supporting_spd,gap_time_seconds,gap_distance_meters,required_speed_mps,gap_bearing,ref_match_exact,is_same_source,speed_diff_A_observ,speed_diff_B_observ,telemetry_speed_diff
0,35.671734,35.601289,-107.336176,-107.553976,feed-a,DC8A69,181.304707,NaN,53.0,35.313578,...,49.000000,553.735,95398.564599,172.281984,250.694544,0,1,9.022723,1.124984,4.000000
1,35.671734,35.601289,-107.336176,-107.553976,feed-a,DC8A69,181.304707,NaN,53.0,35.378941,...,37.750000,324.964,39728.696485,122.255685,231.614246,0,1,59.049023,42.604969,15.250000
2,35.671734,35.601289,-107.336176,-107.553976,feed-a,DC8A69,181.304707,NaN,53.0,35.300919,...,40.500000,319.779,50540.482303,158.048159,228.757163,0,0,23.256549,3.565687,12.500000
3,35.671734,35.601289,-107.336176,-107.553976,feed-a,DC8A69,181.304707,NaN,53.0,35.041899,...,42.875000,558.109,90933.325168,162.931121,227.052662,0,0,18.373586,49.123879,10.125000
4,35.671734,35.601289,-107.336176,-107.553976,feed-a,DC8A69,181.304707,NaN,53.0,35.378185,...,52.916667,318.097,37943.051592,119.281387,229.261724,0,1,62.023320,27.232264,0.083333


In [ ]:
#   Dropping the string datatypes
#   Cols dropped:
#       1. A_source
#       2. A_observ_ref
#       3. B_source
#       4. B_observ_ref


X = X.select_dtypes(exclude=['object', 'string'])
X.info()

<class 'pandas.DataFrame'>
RangeIndex: 216 entries, 0 to 215
Data columns (total 23 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   A_start_lat           216 non-null    float64
 1   A_end_lat             216 non-null    float64
 2   A_start_lon           216 non-null    float64
 3   A_end_lon             216 non-null    float64
 4   A_observ_spd          138 non-null    float64
 5   A_observ_hdg          78 non-null     float64
 6   A_supporting_spd      216 non-null    float64
 7   B_start_lat           216 non-null    float64
 8   B_end_lat             216 non-null    float64
 9   B_start_lon           216 non-null    float64
 10  B_end_lon             216 non-null    float64
 11  B_observ_spd          150 non-null    float64
 12  B_observ_hdg          66 non-null     float64
 13  B_supporting_spd      216 non-null    float64
 14  gap_time_seconds      216 non-null    float64
 15  gap_distance_meters   216 non-null